In [ ]:
# 1. 複製 ComfyUI 官方儲存庫並安裝核心依賴
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -r requirements.txt

In [ ]:
# 2. 安裝額外建議的依賴（加速與管理使用）
!pip install xformers

In [ ]:
# 3. 自動化下載 Z-Image Turbo 相關模型檔案
print("開始下載模型，檔案較大請耐心等候...")

# 3.1 改下載 FP4 量化版的主模型
!wget -O /content/ComfyUI/models/diffusion_models/z_image_turbo_nvfp4.safetensors \
"https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_nvfp4.safetensors"

# 3.2 下載 FP4 量化版的文字編碼器
!wget -O /content/ComfyUI/models/text_encoders/qwen_3_4b_fp4_mixed.safetensors \
"https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b_fp4_mixed.safetensors"

# 3.3 下載 VAE 解碼器
!wget -O /content/ComfyUI/models/vae/ae.safetensors \
"https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors"

print("所有模型下載完成！")

In [ ]:
# 4. 安裝雲端穿透工具（讓你可以點擊連結開啟 ComfyUI 網頁介面）
!pip install pycloudflared

In [ ]:
# 5. 啟動 ComfyUI 並透過 Cloudflared 輸出公開網址（加上低顯存優化參數）
import subprocess
import threading
import time
import socket
import sys

def iframe_thread(port):
    # 1. 等待本機 Port 開啟
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break
            
    print("\n[系統提示] ComfyUI 已成功啟動！正在產生 Cloudflare 穿透連結...\n")
    
    # 2. 同時導向 stdout 和 stderr，確保一定抓得到輸出
    p = subprocess.Popen(
        ["pycloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], 
        stdout=subprocess.PIPE, 
        stderr=subprocess.STDOUT,  # 關鍵：把錯誤輸出融合進標準輸出
        text=True
    )
    
    # 3. 讀取並顯示連結
    for line in iter(p.stdout.readline, ""):
        if "trycloudflare.com" in line:
            # 整理一下字串，只把網址那一行漂亮地印出來
            import re
            url = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if url:
                print(f"👉 你的 ComfyUI 傳送門: {url.group(0)} 👈\n")
            else:
                print(line.strip())

# 啟動背景執行緒
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

In [ ]:
# 執行 ComfyUI 主程式（加上跨網域允許參數與低顯存優化）
!python main.py --dont-upcast-attention --lowvram --port 8188 --enable-cors-header '*'